# End-to-End ML on Snowflake: Customer Churn

This notebook is a complete, runnable ML pipeline with synthetic data, no external dependencies, and no compute pool configuring. This way it works in any account, including a fresh trial account!

We're looking to answer the following question today: Given a customer's behaviour as of an observation date, will they churn in the **following 90 days**?

## What We're Building

| § | Step | Snowflake feature |
|---|------|-------------------|
| 1 | Database, schema, warehouse | SQL DDL |
| 2 | Generate + land 50K customers | `write_pandas` |
| 3 | Leakage scan + baseline | — |
| 4 | Define features once, reusably | **Feature Store** |
| 5 | Snapshot the training data | **Datasets** |
| 6 | Train, logging every run | **Experiment Tracking** |
| 7 | Version the model | **Model Registry** |
| 8 | Score from Python and SQL | **Batch inference** |
| 9 | Watch for drift and decay | **Model Monitor** |
| 10 | Run it on a schedule | **Task Graph (DAG)** |
| 10b | Retrain safely, behind a gate *(optional)* | Tasks + Registry aliases |
| 11 | Trace what fed what | **ML Lineage** |
| 12 | Unified pipeline observability | Task history + DESCRIBE |

## Requirements

- A role that can create databases and warehouses- `SYSADMIN` works well if you're on a completely new account. You will also need to create a Task or two, but since that permission is only granted by default to ACCOUNTADMINS that part of the lab can be come back to later.

This command can enable that permission, though:

```
USE ROLE ACCOUNTADMIN;
GRANT EXECUTE TASK ON ACCOUNT TO ROLE <your_role>;
```

- No packages are required to be installed separately! Snowflake partners with Anaconda to provide native support for all commonly-used Python packages. Everything needed here is available by default in a Snowflake Notebook.

- No compute pools are required since every cell runs on a standard virtual warehouse. Compute pools are only needed for GPU training, distributed workloads, or real-time inference endpoints. None of those apply here since we're only dealing with 50k customers of dummy data.

Runtime is roughly 5–7 minutes total! The Feature Store and Model Monitor steps create dynamic tables that refresh in the background, so a couple of cells say "wait and re-run". This is expected, not a failure.

## A note on how this is written

Every cell is Python. SQL runs through `session.sql(...)` rather than native SQL cells, so the
notebook behaves identically in a Snowflake Notebook and on a local kernel.

---
## 1. Setup

- Everything lives inside the DB and schema we'll create, `ML_DEMO.CHURN`, so cleanup at the end is pretty simple!

- An XSMALL warehouse here is plenty, so feel free to stick with whatever your default warehouse is.

- We're only working with 50K rows since we're here to learn the workflow, not to stress test the compute.

In [ ]:
%%sql -r dataframe_2
-- DON'T WORRY IF YOU CAN'T DO THIS! IT'S FOR A VERY SMALL PART OF THE LAB!!!

USE ROLE ACCOUNTADMIN;
GRANT CREATE DATABASE ON ACCOUNT TO ROLE SYSADMIN;
GRANT CREATE WAREHOUSE ON ACCOUNT TO ROLE SYSADMIN;
GRANT EXECUTE TASK ON ACCOUNT TO ROLE SYSADMIN;
GRANT CREATE INTEGRATION ON ACCOUNT TO ROLE SYSADMIN;


In [ ]:
##--- IF THIS CELL FAILS, CHECK THAT YOUR ROLE HAS THE NECESSARY PERMISSIONS!!! ----------------------

import numpy as np
import pandas as pd

from snowflake.snowpark.context import get_active_session

session = get_active_session()

##--------- EDIT TO YOUR SPECIFIC NEEDS ------------------

# Setting the values for our DB, Schema, and Warehouse- 
DB, SCHEMA, WAREHOUSE = "ML_DEMO", "CHURN", "ML_DEMO_WH"

##--------------------------------------------------------

FQ = f"{DB}.{SCHEMA}" # fully-qualified schema prefix

# Setting the name and version we want to store our trained model as later
MODEL_NAME, MODEL_VERSION = "CHURN_CLASSIFIER", "V1"

setup_statements = [
    # SYSADMIN is the usual choice in a trial account. Change if your account differs.
    "USE ROLE SYSADMIN",
    f"CREATE DATABASE IF NOT EXISTS {DB}",
    f"CREATE SCHEMA IF NOT EXISTS {FQ}",
    f"""CREATE WAREHOUSE IF NOT EXISTS {WAREHOUSE}
            WAREHOUSE_SIZE = 'XSMALL'
            AUTO_SUSPEND   = 60
            AUTO_RESUME    = TRUE
            INITIALLY_SUSPENDED = TRUE""",
    f"USE WAREHOUSE {WAREHOUSE}",
    f"USE SCHEMA {FQ}",
]

for stmt in setup_statements:
    session.sql(stmt).collect()

print(f"Role      : {session.get_current_role()}")
print(f"Database  : {session.get_current_database()}")
print(f"Schema    : {session.get_current_schema()}")
print(f"Warehouse : {session.get_current_warehouse()}")

---
## 2. Generate and land the data

Real projects start with SQL joins across order, billing, and support tables. However, since we've had several webinars cover that recently, we're skipping that here and creating dummy data. 

- **Structure**: One row per customer per observation date. That is the shape you
always want to reach before modelling.

- **Pattern**: Hidden linear relationship plus deliberate noise (the signal is real but messy and imperfect)

- **Expected result**: A well-behaved model should land around 0.78-0.85 AUC. If you ever see 0.99 on a problem like this, suspect leakage rather than celebrating.

Note that here we *know* that there's a relationship hidden in our data. In a real scenario you'll often need to do some exploratory data analysis, sometimes including some variety of regression, to determine this.

I *personally* would recommend these resources to get started with EDA, though please note these aren't official Snowflake recommendations:

- *Why EDA*: <https://online.hbs.edu/blog/post/exploratory-data-analysis>
  
- *A Thorough Step-by-Step*: <https://www.geeksforgeeks.org/data-analysis/what-is-exploratory-data-analysis/>

- *Relating EDA concepts and eventual ML needs*: <https://medium.com/@post.gourang/a-holistic-guide-to-exploratory-data-analysis-eda-for-machine-learning-and-deep-learning-bc4f18f0143b>

In [ ]:
rng = np.random.default_rng(42)   # fixed seed -> reproducible demo
N = 50_000

# Observation dates spread over 12 months. We need a time axis in order to split by time
# later, and so Model Monitor has something to trend against.
start = pd.Timestamp("2025-01-01")
obs_date = start + pd.to_timedelta(rng.integers(0, 365, N), unit="D")

plan   = rng.choice(["BASIC", "PLUS", "PREMIUM"], N, p=[0.55, 0.30, 0.15])
region = rng.choice(["NORTH", "SOUTH", "EAST", "WEST"], N)

tenure_months         = rng.integers(1, 61, N)
monthly_spend         = np.round(rng.gamma(shape=3.0, scale=28.0, size=N) + 10, 2)
orders_30d            = rng.poisson(lam=np.clip(monthly_spend / 45, 0.2, 12), size=N)
days_since_last_order = rng.integers(0, 121, N)
support_tickets_90d   = rng.poisson(0.7, N)
discount_rate         = np.round(rng.beta(2, 8, N), 3)
avg_order_value       = np.round(monthly_spend / np.clip(orders_30d, 1, None), 2)

# --- Hidden truth: what actually drives churn -------------------------------------
# Long gaps since the last order and support friction push churn UP.
# Tenure, spend, order frequency, and a richer plan push it DOWN.
logit = (
    -1.35
    + 0.0180 * days_since_last_order
    + 0.2600 * support_tickets_90d
    - 0.0230 * tenure_months
    - 0.0060 * monthly_spend
    - 0.1400 * orders_30d
    + 1.1000 * discount_rate
    + np.where(plan == "BASIC", 0.45, np.where(plan == "PLUS", 0.0, -0.40))
    + rng.normal(0, 0.85, N)          # irreducible noise -- keeps AUC realistic
)
churned = (1 / (1 + np.exp(-logit)) > rng.uniform(0, 1, N)).astype(int)

customers = pd.DataFrame({
    "CUSTOMER_ID":           np.arange(1, N + 1),
    "OBSERVATION_DATE":      obs_date,
    "PLAN_TYPE":             plan,
    "REGION":                region,
    "TENURE_MONTHS":         tenure_months,
    "MONTHLY_SPEND":         monthly_spend,
    "ORDERS_LAST_30D":       orders_30d,
    "DAYS_SINCE_LAST_ORDER": days_since_last_order,
    "SUPPORT_TICKETS_90D":   support_tickets_90d,
    "AVG_ORDER_VALUE":       avg_order_value,
    "DISCOUNT_RATE":         discount_rate,
    "CHURNED":               churned,
})

print(f"{len(customers):,} rows generated")
print(f"Churn rate: {customers['CHURNED'].mean():.1%}")
customers.head()

In [ ]:
# Land it in Snowflake. From here on, the TABLE is the source of truth, not the DataFrame- you'll see why soon.
session.write_pandas(
    customers,
    table_name="CUSTOMER_CHURN",
    database=DB,
    schema=SCHEMA,
    auto_create_table=True,
    overwrite=True,
)

# The Feature Store builds a dynamic table on top of this, which requires change tracking.
# Enabling it up front avoids a confusing error two sections from now.
session.sql(f"ALTER TABLE {FQ}.CUSTOMER_CHURN SET CHANGE_TRACKING = TRUE").collect()

print(f"Wrote {FQ}.CUSTOMER_CHURN")
session.table("CUSTOMER_CHURN").show(5)

---
## 3. Sanity checks and a baseline

Two checks that cost almost nothing and that many people routinely forget:

- **Leakage scan:** Score each feature *on its own* as a predictor. A single feature reaching AUC
above ~0.95 almost never means you found a brilliant signal- it means you found the answer key that we'd normally be using to score the models, not train them! For example, A `CANCELLATION_DATE` column would light up here instantly.

- **Baseline:** Predict the majority class for every customer. This model has zero training or tuning, and its
score is the bar every real model must clear. Without it you cannot tell whether 0.82 AUC is good
or embarrassing.

Watch the accuracy the baseline achieves- it looks good, sure, but it only represents one aspect of your model, and should always be considered alongside other metrics as well, like AUC and average precision! Otherwise, you're likely to have a model that looks great during training but fails its field tests.

In [ ]:
from sklearn.metrics import roc_auc_score

# Column groups, defined once and reused everywhere downstream.
# Separated by type so it's easier to maintenance down the road!
CATEGORICAL = ["PLAN_TYPE", "REGION"]
NUMERIC = [
    "TENURE_MONTHS", "MONTHLY_SPEND", "ORDERS_LAST_30D", "DAYS_SINCE_LAST_ORDER",
    "SUPPORT_TICKETS_90D", "AVG_ORDER_VALUE", "DISCOUNT_RATE",
]

# Creates a complete list of all of our features (what we want to use to predict things)
FEATURES = CATEGORICAL + NUMERIC

# Sets our target column (what we want to predict)
TARGET = "CHURNED"

# Defining the "answer key"
df = session.table("CUSTOMER_CHURN").to_pandas()
y_all = df[TARGET]

# Printing some initial data statistics
print("Class balance")
print(y_all.value_counts().rename({0: "retained", 1: "churned"}).to_string())
print(f"  positive rate : {y_all.mean():.1%}")
print(f"  missing values: {int(df.isna().sum().sum())}\n")

# --- Single-feature leakage scan ---------------------------------------------------
# Note that the CHURNED column isn't here since that's what we're trying to predict!

print("Single-feature AUC  (>0.95 would be a leakage red flag)")
for c in NUMERIC:
    auc = roc_auc_score(y_all, df[c])
    # A feature can leak by being strongly NEGATIVELY predictive too, hence max(auc, 1-auc).
    flag = "   <-- SUSPICIOUS" if max(auc, 1 - auc) > 0.95 else ""
    print(f"  {c:<24} {auc:.3f}{flag}")

# --- Baseline: always predict the majority class ----------------------------------
# Same note here- including CHURNED would be like giving the model the answer key!

majority = int(y_all.mode()[0])
print(f"\nBaseline (always predict {majority}):")
print(f"  accuracy = {(y_all == majority).mean():.3f}   <-- looks respectable, model is useless")
print(f"  AUC      = 0.500           <-- the honest number. This is the bar to beat.")

---
## 4. Feature Store

The problem this solves is subtle and expensive. Doing it manually, you'd compute "orders in the last 30 days" one way in your training notebook, and a production job might compute it slightly differently. 

It's tricky to keep those versions the same with so much training and testing going on for a new model you're still optimizing, and if the numbers disagree, the model quietly degrades. That is **train/serve skew**, and it is very hard to notice because nothing errors.

--

The Snowflake Feature Store fixes it structurally: features are defined **once**, in Snowflake, and both training and serving read that one definition.

Three concepts:

- **Entity**: what a feature describes, and the key used to join it. Here, we have a customer in `CUSTOMER_ID`.
  
- **Feature View**: the feature definitions themselves, as a query. With `refresh_freq` set it becomes a *managed* feature view, backed by a dynamic table. 

  - Note that features are *defined* here, not cleaned. If `CUSTOMER_CHURN` were to receive new malformed data, the Feature View would faithfully serve that malformed data. Cleaning belongs in the query that defines the Feature View, not in a separate step.
    
  - We will also adjust how the Feature refreshes later on, but we'll talk more about that in a later section!
  
- **Training set**: the join of a *spine* (the list of events you want to predict, in this case who, when, and the label) against the feature views. The join is **point-in-time correct**: each row gets features as they were on that row's date, not as they are today.

Note that the label `CHURNED` is deliberately excluded from the feature view, since labels are not features.

In [ ]:
from snowflake.ml.feature_store import CreationMode, Entity, FeatureStore, FeatureView

# Registers the schema we're using from the Setup section as a Feature Store
fs = FeatureStore(
    session=session,
    database=DB,
    name=SCHEMA,
    default_warehouse=WAREHOUSE,
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)

# --- Entity: what the features describe, and how to join to it ---------------------
# Creating these is best practice so that you don't have to write complicated joins later on

customer = Entity(
    name="CUSTOMER",
    join_keys=["CUSTOMER_ID"],
    desc="A subscription customer, keyed on CUSTOMER_ID",
)

# Register it to our Feature Store
fs.register_entity(customer)

# --- Feature View: the feature definitions, as a query ----------------------------
# Note CHURNED is absent. Labels are not features.

feature_df = session.table("CUSTOMER_CHURN").select(
    "CUSTOMER_ID", "OBSERVATION_DATE", *FEATURES
)

# Creates a Feature View here that automatically refreshes every day
customer_fv = FeatureView(
    name="CUSTOMER_FEATURES",
    entities=[customer],
    feature_df=feature_df,
    timestamp_col="OBSERVATION_DATE",   # enables point-in-time lookups
    refresh_freq= "1 day",              # managed -> backed by a dynamic table
    desc="Behavioural and account features for churn prediction",
)

# Register the View to the Feature Store
customer_fv = fs.register_feature_view(
    feature_view=customer_fv,
    version="V1",
    overwrite=True,
)

# Hand refresh control to the task graph. Without this the dynamic table keeps its own
# 1-day schedule AND gets refreshed by the DAG -- two schedulers, no coordination.
session.sql(f"ALTER DYNAMIC TABLE {customer_fv.fully_qualified_name()} SET SCHEDULER = DISABLE").collect()
print("Feature view scheduler disabled -- refreshes now come only from CHURN_PIPELINE")

print("Registered feature view CUSTOMER_FEATURES/V1")
fs.list_feature_views().show()

# Ignore the warning here about ipywidgets, we're fine without it.

In [ ]:
# Build the training set from the Feature Store.

# generate_training_set joins the spine (who, when, what happened) to the feature view
# AS OF each row's OBSERVATION_DATE, that way you get point-in-time correctness. This is the step that
# prevents leakage through the feature path: every row gets features as they were on that
# date, not as they are today.

# The label (CHURNED) lives in the spine, not in the feature view. 
# Labels are not features, and should never be included in training datasets!

# Note: we also include OBSERVATION_DATE here so we can do the time-based split later.

spine_df = session.table("CUSTOMER_CHURN").select("CUSTOMER_ID", "OBSERVATION_DATE", TARGET)

training_sdf = fs.generate_training_set(
    spine_df=spine_df,
    features=[customer_fv],
    spine_timestamp_col="OBSERVATION_DATE",
    spine_label_cols=[TARGET],
)

# Printing simple metrics
print(f"Training set: {training_sdf.count():,} rows")
print("Columns:", training_sdf.columns)
training_sdf.show(5)

---
## 5. Dataset Creation: Snapshot the training data

A Dataset is an **immutable, versioned snapshot** of the training set we just built.

Why bother, when the training set is one query away? Because that query won't return the same rows next month. The source table gets new data, someone backfills a correction, a feature definition is amended. Six months from now, "what exactly did V1 train on?" becomes unanswerable without a snapshot.

This section will sweep those worries right away, since `V1` will be frozen. It also creates a lineage edge from the data to the model, which we'll query in Section 11!

In [ ]:
from snowflake.ml import dataset

# Setting the name and version number for later!
DATASET_NAME, DATASET_VERSION = "CHURN_TRAINING_DATA", "V1"

# Ensure session context is on the schema we made earlier (can drift after deletes/creates).
session.sql(f"USE SCHEMA {FQ}").collect()

# Versions are immutable, so a re-run must drop the old one first.
# Drop the entire dataset (not just the version) to avoid orphaned metadata.
try:
    ds = dataset.load_dataset(session, f"{FQ}.{DATASET_NAME}", DATASET_VERSION)
    ds.delete()
    print(f"Removed existing dataset {DATASET_NAME}")
except Exception:
    pass

# Materialize the training set first -- the Feature Store query plan is too complex
# for create_from_dataframe to handle directly, causing an internal SQL error.
training_sdf_cached = training_sdf.cache_result()

# Creating the dataset from the cache, since otherwise we run into some SQL limitations
churn_dataset = dataset.create_from_dataframe(
    session,
    name=f"{FQ}.{DATASET_NAME}",
    version=DATASET_VERSION,
    input_dataframe=training_sdf_cached,
)

print(f"Created dataset {FQ}.{DATASET_NAME}/{DATASET_VERSION}  (immutable)")

# Read back via Snowpark instead of the Ray-based reader, which can hit
# FileNotFoundError on internal-stage parquet files in notebook environments.
train_full = training_sdf_cached.to_pandas()
print(f"Read back {len(train_full):,} rows")
train_full.head()

---
## 6. Split, then train

There are two major rules, and both are routinely ignored since it's rare to find step-by-step guides like this:

- **Split before you preprocess.** Scalers and encoders are fitted on the training set *only*. Fit
them on everything and information from the test set bleeds into training, making your test score
fiction.

- **Split by time, not at random.** In production you always predict forward. A random split lets the
model learn from the future, which inflates your score and hides real decay. We split at the
**80th percentile of observation dates** — approximately the first 10 months of data in training
and the most recent ~20% in the holdout.

Here we'll be placing all of our preprocessing steps inside a nifty tool from Scikit-learn called a `Pipeline`. This effectively makes the process into the Python equivalent of a Stored Procedure, so that every time you run your model all of its component parts run identically.

This combined with the Feature Store is also how we can largely prevent train/serve skew, since we've blocked its rise in both places it can show up (feature computation and transformation).

In [ ]:
train_full["OBSERVATION_DATE"] = pd.to_datetime(train_full["OBSERVATION_DATE"])

# Chronological split at the 80th percentile of dates.
CUTOFF = train_full["OBSERVATION_DATE"].quantile(0.80)

# Set the train set before the cutoff, test to be after
train_df = train_full[train_full["OBSERVATION_DATE"] <= CUTOFF]
test_df  = train_full[train_full["OBSERVATION_DATE"] >  CUTOFF]

# Same thing for the features and target columns
X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_test,  y_test  = test_df[FEATURES],  test_df[TARGET]

print(f"Cutoff : {CUTOFF.date()}")
print(f"Train  : {len(X_train):>6,} rows  "
      f"({train_df['OBSERVATION_DATE'].min().date()} -> {train_df['OBSERVATION_DATE'].max().date()})")
print(f"Test   : {len(X_test):>6,} rows  "
      f"({test_df['OBSERVATION_DATE'].min().date()} -> {test_df['OBSERVATION_DATE'].max().date()})")
print(f"Churn  : train {y_train.mean():.1%} | test {y_test.mean():.1%}")

### Training, with experiment tracking

We train two models here (logistic regression and gradient boosting trees) and log each as a **run**. 

- Before we look at the results, we can make guesses on which one is better, but we won't know until we look at the numbers. Just because GBoost models are newer doesn't mean they'll always be better!

- Tracking experiments is important, because doing it by hand in a notebook or Excel sheet is miserable! Once you've tried a dozen variants over two weeks, you're going to want the parameters and metrics in a queryable table like we'll set up here. Plus, this solution only adds three lines per run.

Note that the logistic regression model gets `StandardScaler` and the tree does not because trees don't really get affected by scaling like this one! This is why we build separate preprocessing pipelines per model rather than sharing one generic step. 

--

More details, for those who are curious: Trees only use feature *order* when deciding where to split, so rescaling the values has no effect on them. Linear models, on the other hand, weight each feature directly, so unequal scales can cause the regularizer to unfairly penalise features that happen to be measured in larger units.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from snowflake.ml.experiment import ExperimentTracking

# Make a replicable pipeline builder that distinguishes between our models
def build_pipeline(estimator, scale: bool) -> Pipeline:
    """Preprocessing + model as one object. Everything is fitted on train only."""
    numeric_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale: #if "scale" is requested then we put in the StandardScaler as needed
        numeric_steps.append(("scale", StandardScaler()))
        
    #puts in our transformation steps so we don't have to redo those by hand every time
    pre = ColumnTransformer([
        ("num", Pipeline(numeric_steps), NUMERIC),
        # handle_unknown="ignore" -> a plan tier invented after training will not crash inference.
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    ])
    return Pipeline([("pre", pre), ("model", estimator)])


# --- CHANGE THIS TO EXPERIMENT WITH MODEL PARAMS ---------------------------------
GB_PARAMS = dict(max_iter=200, learning_rate=0.01, early_stopping=False, random_state=42)
# GB_PARAMS = dict(max_iter=200,  learning_rate=0.08, early_stopping=False, random_state=42)
# GB_PARAMS = dict(max_iter=200,  learning_rate=0.50, early_stopping=False, random_state=42)
# GB_PARAMS = dict(max_iter=2000, learning_rate=0.01, early_stopping=False, random_state=42)

LR_PARAMS = dict(max_iter=1000, class_weight="balanced")

#----------------------------------------------------------------------------------

# Run names must be valid SQL identifiers (no dots), so replace '.' with 'p'.
gb_run_name = f"gradient_boosting_lr{str(GB_PARAMS['learning_rate']).replace('.', 'p')}_it{GB_PARAMS['max_iter']}"

# Lists the models we're looking at, relies on the params specified earlier
candidates = {
    "logistic_regression": (
        build_pipeline(LogisticRegression(**LR_PARAMS), scale=True),
        {"model": "LogisticRegression", **LR_PARAMS},
    ),
    gb_run_name: (
        build_pipeline(HistGradientBoostingClassifier(**GB_PARAMS), scale=False),
        {"model": "HistGradientBoostingClassifier", **GB_PARAMS},
    ),
}

# Sets our Experiment object in the schema so we can track our runs
# Force a fresh instance in case the singleton is stale from a previous run
ExperimentTracking._instance = None
exp = ExperimentTracking(session=session, database_name=DB, schema_name=SCHEMA)

# CHANGE THIS NAME WITH SUBSEQUENT RUNS, LOGGED AS HISTORICAL DATA
exp.set_experiment("CHURN_EXPERIMENT_V1")

# Runs the experiment on both models
results = {}
for name, (pipe, params) in candidates.items():
    with exp.start_run(run_name=name):
        pipe.fit(X_train, y_train)
        proba = pipe.predict_proba(X_test)[:, 1]

        # Calculating AUC and avg precision here
        auc = roc_auc_score(y_test, proba)
        ap = average_precision_score(y_test, proba)

        # Logging all the metrics for later
        exp.log_params({**params, "train_rows": len(X_train), "cutoff_date": str(CUTOFF.date())})
        exp.log_metrics({"test_roc_auc": float(auc), "test_avg_precision": float(ap)})

        results[name] = {"auc": auc, "ap": ap}
        print(f"{name:<22} AUC={auc:.4f}  AP={ap:.4f}")

# Selects a "champion", aka the best model run and saves its results
champion_name = max(results, key=lambda k: results[k]["auc"])
champion = candidates[champion_name][0]
champion_auc, champion_ap = results[champion_name]["auc"], results[champion_name]["ap"]

# Prints the results of the champion
print(f"\nChampion: {champion_name}")
print(f"  AUC {champion_auc:.4f} vs baseline 0.500  ->  lift {champion_auc - 0.5:+.4f}\n")
print(classification_report(y_test, champion.predict(X_test),
                            target_names=["retained", "churned"]))

---
## 7. Model Registry

Here, we'll store versions of our models so we can come back to them later! Registering does four things at once:

- **Versions the model**, so `V1` still exists after you train `V2`
- **Records input columns and types**, inferred from `sample_input_data`
- **Captures the environment**, so the model can be reloaded and reproduced
- **Makes it callable from SQL**, which is what turns inference into an ordinary query

Two arguments are easy to omit and painful to have omitted:

- `sample_input_data`: without it there is no schema, and inference cannot validate its inputs
- `task`: Model Monitor needs it to know which metrics apply to what models. Skip it and you lose your drift detection!

Setting `default` gives you a modular pointer to whatever your current best model is- that way you don't have to hardcode "V1" into so many places and edit it every time you want to experiment!

In [ ]:
from snowflake.ml.model import task as ml_task
from snowflake.ml.registry import Registry

# Assign a Registry to the Schema- typically the Registry and Feature Store use different schemas 
# for governance reasons, but since we're playing around here I haven't separated them out.
reg = Registry(session=session, database_name=DB, schema_name=SCHEMA)

# Re-running the notebook would collide on the same version name, so clear it first.
try:
    reg.get_model(MODEL_NAME).delete_version(MODEL_VERSION)
    print(f"Removed existing {MODEL_NAME}/{MODEL_VERSION}")
except Exception:
    pass

# Logs the details about the run that just happened so we have records on what we did
mv = reg.log_model(
    model=champion,                    # the whole Pipeline -- preprocessing travels with it
    model_name=MODEL_NAME,
    version_name=MODEL_VERSION,
    sample_input_data=training_sdf_cached.select(FEATURES).limit(100),
    task=ml_task.Task.TABULAR_BINARY_CLASSIFICATION,
    comment=f"90-day churn model ({champion_name}); time-based split at {CUTOFF.date()}",
    target_platforms=["WAREHOUSE"],
)

# Metrics live with the version, so V1 and V2 can be compared later.
mv.set_metric("test_roc_auc", float(champion_auc))
mv.set_metric("test_avg_precision", float(champion_ap))
mv.set_metric("train_rows", int(len(X_train)))

# A moving pointer -- downstream code targets the default, not a hardcoded version.
reg.get_model(MODEL_NAME).default = MODEL_VERSION

print(f"\nRegistered {FQ}.{MODEL_NAME}/{MODEL_VERSION}")
print("Callable functions:", [f["name"] for f in mv.show_functions()])
reg.show_models()

# Don't worry about these warnings, they're optional implementation pieces we don't need at this moment

---
## 8. Batch inference

Analyzing data in bulk makes up the vast majority of ML implementations I see today, because for the most part those needs involve scoring a table of results, writing those results to a dashboard, and then perhaps a person comes along every so often and reviews that dashboard. 

So, because this webinar is geared towards folks with relatively little ML experience we won't be covering real-time deployments in great detail today, but if you want more info on how that works please refer to the seminars from my teammates that I've included in the slides! They did some great in-depth work. 

In this section, we'll be passing a **Snowpark** DataFrame to `mv.run()`, so the data never leaves Snowflake and never enters this notebook's memory. That is what lets the same code work on 50 thousand rows or 50 million, and while we'll explore automating these runs later for now I just want you to see how it's done by hand. 

--

Note that `predict_proba` returns one column per class: 

- `output_feature_0` is the probability a customer will be retained
- `output_feature_1` is the probability a customer will churn.

In [ ]:
from snowflake.snowpark.functions import call_builtin, cast, col, lit, to_date

# Set the cutoff period for later
CUTOFF_STR = CUTOFF.strftime("%Y-%m-%d")

# OBSERVATION_DATE is stored as NUMBER(38,0) epoch nanoseconds by write_pandas;
# use scale=9 to convert correctly to TIMESTAMP_NTZ.
def ts_ntz_nanos(c):
    return call_builtin("to_timestamp_ntz", c, lit(9))

# Treat the holdout period as "new" data arriving to be scored.
scoring_sdf = session.table("CUSTOMER_CHURN").filter(
    ts_ntz_nanos(col("OBSERVATION_DATE")) > to_date(lit(CUTOFF_STR))
)
print(f"Scoring {scoring_sdf.count():,} rows")

# Setting the run as the "scored" one
scored = mv.run(scoring_sdf, function_name="predict_proba")

# These probabilities are NOT calibrated -- P=0.70 does NOT mean 70% will churn.
# It means that one customer is more likely than another at, say P=0.3, to churn!
# The reason we didn't do it here is because that would have required a third split of our already small dataset.

# For ranking (top-N campaigns) this is fine. For absolute probabilities (pricing,
# clinical risk), use this package to CALIBRATE the probabilities FIRST:
# https://scikit-learn.org/stable/modules/generated/sklearn.calibration.CalibratedClassifierCV.html

predictions = scored.select(
    col("CUSTOMER_ID"),
    col("OBSERVATION_DATE"),
    col("CHURNED").alias("ACTUAL_CHURNED"),
    cast(col('"output_feature_1"'), "FLOAT").alias("CHURN_PROBABILITY"),
)

# Saves the probabilities as a table for later use
predictions.write.save_as_table("CHURN_PREDICTIONS", mode="overwrite")

print(f"Wrote {FQ}.CHURN_PREDICTIONS")
session.table("CHURN_PREDICTIONS").order_by(col("CHURN_PROBABILITY").desc()).show(10)

In [ ]:
# --- The same inference, from pure SQL --------------------------------------------
# This is the part that tends to surprise people: because the model is a registry object,
# an analyst who has never touched Python can use it, and it drops straight into a view,
# a dynamic table, or a dbt model.
#
#   <model>!PREDICT_PROBA(...)   calls one of the model's registered functions

sql_inference = f"""
SELECT
    CUSTOMER_ID,
    PLAN_TYPE,
    TENURE_MONTHS,
    DAYS_SINCE_LAST_ORDER,
    SUPPORT_TICKETS_90D,
    ROUND({FQ}.{MODEL_NAME}!PREDICT_PROBA(
        PLAN_TYPE, REGION, TENURE_MONTHS, MONTHLY_SPEND, ORDERS_LAST_30D,
        DAYS_SINCE_LAST_ORDER, SUPPORT_TICKETS_90D, AVG_ORDER_VALUE, DISCOUNT_RATE
    ):output_feature_1::FLOAT, 4) AS CHURN_PROBABILITY
FROM {FQ}.CUSTOMER_CHURN
WHERE CHURN_PROBABILITY > 0.75
ORDER BY CHURN_PROBABILITY DESC
LIMIT 15
"""

print("Highest-risk customers, computed entirely in SQL:")
session.sql(sql_inference).show()

---
## 9. Model Monitor

Even the best models decay. Customer behaviour can shift, an upstream pipeline changes their units, and then your model quietly gets worse. You want to learn that from a dashboard, not from a stakeholder who's upset that you results made them look silly!

Thus, the monitor we'll create today needs one table containing, per prediction:

| Item | Column Name(s) | Why it is needed |
|-----------|-------------|------------------|
| ID | `CUSTOMER_ID` | join predictions to outcomes |
| Timestamp | `PREDICTION_TIMESTAMP` | the axis every metric trends over |
| Prediction score | `CHURN_PROBABILITY` | what the model said |
| Actual outcome | `ACTUAL_CHURNED` | ground truth |
| **Input features** | the feature columns | required for **drift** detection |

Including the features is what separates real monitoring from a scorecard. Performance metrics can tell
you the model got worse, and feature drift can tell you *which input changed*, meaning once this is set you'll have the tools to figure out the difference between "retrain" and "go fix the pipeline that started sending days as hours".

The baseline we'll take below is the reference distribution drift is measured against, since it's what the model considers "normal".

--

**Note here though:** here we have a "perfect" view of our data, where each customer/ID/prediction has an answer. In a real production environment this wouldn't necessarily be the case, since you wouldn't know 90-day churn for 90 days. 

Drift metrics will typically be available instantly, sure, but performance metrics will usually have to backfill as outcomes (churn/not churn in this case) arrive. 

In [ ]:
# --- Source table: features + prediction + actual + timestamp, all in one place ----
(
    scored
    .with_column("PREDICTION_TIMESTAMP", ts_ntz_nanos(col("OBSERVATION_DATE")))
    .with_column("CHURN_PROBABILITY", cast(col('"output_feature_1"'), "FLOAT"))
    .with_column("ACTUAL_CHURNED", cast(col("CHURNED"), "FLOAT"))
    .select("CUSTOMER_ID", "PREDICTION_TIMESTAMP", "CHURN_PROBABILITY", "ACTUAL_CHURNED", *FEATURES)
    .write.save_as_table("CHURN_MONITOR_SOURCE", mode="overwrite")
)

# --- Baseline table: the training period, i.e. what "normal" looks like -------------
baseline_scored = mv.run(
    session.table("CUSTOMER_CHURN").filter(
        ts_ntz_nanos(col("OBSERVATION_DATE")) <= to_date(lit(CUTOFF_STR))
    ),
    function_name="predict_proba",
)
(
    baseline_scored
    .with_column("CHURN_PROBABILITY", cast(col('"output_feature_1"'), "FLOAT"))
    .with_column("ACTUAL_CHURNED", cast(col("CHURNED"), "FLOAT"))
    .select("CHURN_PROBABILITY", "ACTUAL_CHURNED", *FEATURES)
    .write.save_as_table("CHURN_MONITOR_BASELINE", mode="overwrite")
)

print(f"CHURN_MONITOR_SOURCE   : {session.table('CHURN_MONITOR_SOURCE').count():>6,} rows")
print(f"CHURN_MONITOR_BASELINE : {session.table('CHURN_MONITOR_BASELINE').count():>6,} rows")
session.table("CHURN_MONITOR_SOURCE").show(5)

In [ ]:
# Here we set the AGGREGATION_WINDOW to 7 days due to how small of a sample size we have
# a daily buckets would be (~130 data points)- with 20% of our data churning, a daily 
# bucket causes any monitoring values to swing wildly, making constant false alarms.

# Creates a monitor based off of our baseline from above, refreshes every 
create_monitor = f"""
CREATE OR REPLACE MODEL MONITOR {FQ}.CHURN_MONITOR
WITH
    MODEL    = {FQ}.{MODEL_NAME}
    VERSION  = {MODEL_VERSION}
    FUNCTION = PREDICT_PROBA
    SOURCE   = {FQ}.CHURN_MONITOR_SOURCE
    BASELINE = {FQ}.CHURN_MONITOR_BASELINE
    ID_COLUMNS               = ('CUSTOMER_ID')
    TIMESTAMP_COLUMN         = 'PREDICTION_TIMESTAMP'
    PREDICTION_SCORE_COLUMNS = ('CHURN_PROBABILITY')
    ACTUAL_CLASS_COLUMNS     = ('ACTUAL_CHURNED')
    WAREHOUSE          = {WAREHOUSE}
    REFRESH_INTERVAL   = '1 hour'
    AGGREGATION_WINDOW = '7 days'
"""
session.sql(create_monitor).collect()

print(f"Created model monitor {FQ}.CHURN_MONITOR")
session.sql(f"SHOW MODEL MONITORS IN SCHEMA {FQ}").show()

In [ ]:
# The monitor's first refresh takes a few minutes. If these return no rows, that is not a
# failure -- wait and re-run this cell.

# In Snowsight these also render as charts under the model version, but everything is
# queryable, which means you can build alerts on it!

# Note that the default granularity is 1 day regardless of aggregation window
GRANULARITY = "7 DAY"     # MUST match AGGREGATION_WINDOW in cell 23 (CREATE MODEL MONITOR)
LOOKBACK_DAYS = 400       # our synthetic dates span 2025, so look back generously
ROWS = 10

# --- Performance: is the model still working? --------------------------------------
# This checks whether the metrics of the model have drifted, and if so, when and by how much?
perf = session.sql(f"""
    SELECT EVENT_TIMESTAMP, METRIC_VALUE, COUNT_USED, COUNT_UNUSED, METRIC_NAME
    FROM TABLE(MODEL_MONITOR_PERFORMANCE_METRIC(
        '{FQ}.CHURN_MONITOR',
        'ROC_AUC',
        '{GRANULARITY}', 
        DATEADD('day', -{LOOKBACK_DAYS}, CURRENT_TIMESTAMP()),
        CURRENT_TIMESTAMP()
    ))
    ORDER BY EVENT_TIMESTAMP DESC
    LIMIT {ROWS}
""").to_pandas()

# DESC in SQL takes the MOST RECENT buckets; flip the order for human-readable display.
perf = perf.sort_values("EVENT_TIMESTAMP")

#In this case, it looks back over the last 10 weeks (the ROWS and GRANULARITY args)
print(f"ROC AUC per {GRANULARITY.lower()} -- most recent {len(perf)} periods")
print(perf.to_string(index=False))

# COUNT_USED is the sample size behind each point, and it is why the window is 7 days.
# At ~130 rows/day (~26 churners) the standard error on AUC is roughly +/-0.05, so
# daily values swing ~0.11 on noise alone -- alerting on that cries wolf constantly.
# A weekly bucket holds ~900 rows and cuts that error by about two thirds.
if not perf.empty:
    print(f"\n  rows per bucket : min {int(perf['COUNT_USED'].min())}, "
          f"max {int(perf['COUNT_USED'].max())}")
    print(f"  AUC range       : {perf['METRIC_VALUE'].min():.4f} - {perf['METRIC_VALUE'].max():.4f}")
    print(f"  skipped rows    : {int(perf['COUNT_UNUSED'].sum())}  (non-zero means null predictions or actuals)")


# --- Drift: which input moved? -----------------------------------------------------
# We have do do SELECT * here on purpose and some name normalization due to some minor items 
# covered by the behaviour-change bundle 2025_04, which isn't something trial accounts will have by default.

# If you want to enable the change bundle in your account to avoid this, please run the following statement:
# ALTER ACCOUNT SET ENABLE_BEHAVIOR_CHANGE_BUNDLE = '2025_04'

# Difference of means measures data before and after an event.
# In this case, it measures any model drift of each week's measurements from baseline.
drift = session.sql(f"""
    SELECT *
    FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
        '{FQ}.CHURN_MONITOR',
        'DIFFERENCE_OF_MEANS', 
        'DAYS_SINCE_LAST_ORDER',
        '{GRANULARITY}',
        DATEADD('day', -{LOOKBACK_DAYS}, CURRENT_TIMESTAMP()),
        CURRENT_TIMESTAMP()
    ))
    ORDER BY EVENT_TIMESTAMP DESC
    LIMIT {ROWS}
""").to_pandas()

# Normalize the two possible column spellings to one.
drift.columns = [
    c.replace("BASELINE_COL_COUNT", "BASELINE_COUNT").replace("COL_COUNT", "COUNT")
    for c in drift.columns
]
wanted = ["EVENT_TIMESTAMP", "METRIC_VALUE", "COUNT_USED", "BASELINE_COUNT_USED", "COLUMN_NAME"]

drift = drift[[c for c in wanted if c in drift.columns]].sort_values("EVENT_TIMESTAMP")
print(f"\n\nDrift on DAYS_SINCE_LAST_ORDER -- most recent {len(drift)} periods")

print(drift.to_string(index=False))

if not drift.empty:
    # DAYS_SINCE_LAST_ORDER is ~uniform 0-120, so std ~35. With ~900 rows per weekly
    # bucket the standard error on the mean is ~1.2 days; anything inside roughly
    # +/-2.5 days is noise. Our synthetic dates carry no time trend, so we EXPECT no
    # drift -- values hovering around zero is the correct result, not a broken monitor.
    print(f"\n  largest absolute shift: {drift['METRIC_VALUE'].abs().max():.2f} days")
    print("  (feature is ~uniform 0-120, std ~35; at ~900 rows/bucket the standard error")
    print("   on the mean is ~1.2 days, so +/-2.5 days is within noise)")

# Drift needs no labels, so it is available immediately. Performance needs outcomes,
# which lag. That asymmetry is why drift is the early-warning signal.

---
## 10. Automation: Task Graph (DAG)

Finally, the part where we get to bring it all together and make it (mostly) run by itself! 

A **Task Graph** is a set of tasks with dependencies, running on a schedule. Ours is deliberately small since we have a smaller dataset, and includes the Feature Store refresh we spoke about earlier:

```
REFRESH_FEATURES  ->  SCORE_CUSTOMERS  ->  REFRESH_MONITOR_SOURCE
```

Or, in order, we refresh our Feature View, run Batch Inference on any new data, and refresh our model monitor. Note that as we mentioned in the Monitoring section you can decouple your monitor from this pipeline if you want it to act more regularly than these other tasks, but here they're combined just so it's a bit more observable.

All of these tasks are also in plain SQL- this isn't the typical loadout, but this means that even if you don't have a compute pool available to you, you should still be able to run this!

### Task mechanics

Two things worth knowing:

- Tasks are created **suspended**. Nothing runs until you resume them so that you don't get any surprises.
- Resume the **root last** and suspend it **first**. Snowflake will not let a child task run without its
  parent, and the root task controls the entire schedule.

We add retraining to the DAG in the next section since we're just looking to get the foundations built here first, so don't worry! I didn't forget, I promise :D

In [ ]:
from datetime import timedelta

from snowflake.core import Root
from snowflake.core.task.dagv1 import DAG, DAGOperation, DAGTask

root = Root(session)

FV_TABLE = customer_fv.fully_qualified_name()

# Task 1: score every customer from the FEATURE VIEW, using whichever model version is
# currently set as DEFAULT. This means EVALUATE_AND_PROMOTE's pointer-move takes effect
# on the very next run -- no code changes needed when promoting.
#
# We use QUALIFY to get one row per customer (their most recent observation).
# Without it the task would score every historical snapshot, not the current state.
#
# ACTUAL_CHURNED is deliberately absent. Predictions are made before outcomes are known;
# a scoring table that contains the label is one someone will eventually train on by mistake.
score_sql = f"""
CREATE OR REPLACE TABLE {FQ}.CHURN_PREDICTIONS AS
WITH churn_model AS MODEL {FQ}.{MODEL_NAME} VERSION DEFAULT
SELECT
    CUSTOMER_ID,
    OBSERVATION_DATE,
    churn_model!PREDICT_PROBA(
        PLAN_TYPE, REGION, TENURE_MONTHS, MONTHLY_SPEND, ORDERS_LAST_30D,
        DAYS_SINCE_LAST_ORDER, SUPPORT_TICKETS_90D, AVG_ORDER_VALUE, DISCOUNT_RATE
    ):output_feature_1::FLOAT AS CHURN_PROBABILITY
FROM (
    SELECT *
    FROM {FV_TABLE}
    QUALIFY ROW_NUMBER() OVER (PARTITION BY CUSTOMER_ID ORDER BY OBSERVATION_DATE DESC) = 1
)
"""

# Task 2: rebuild the monitor source table -- also VERSION DEFAULT so monitoring stays
# in sync with whatever is currently serving. The label is joined from the base table
# because labels are not features and so are not in the feature view.
# No QUALIFY here: monitoring needs the full event history to trend metrics over time.
monitor_sql = f"""
CREATE OR REPLACE TABLE {FQ}.CHURN_MONITOR_SOURCE AS
WITH churn_model AS MODEL {FQ}.{MODEL_NAME} VERSION DEFAULT
SELECT
    f.CUSTOMER_ID,
    f.OBSERVATION_DATE::TIMESTAMP_NTZ AS PREDICTION_TIMESTAMP,
    churn_model!PREDICT_PROBA(
        f.PLAN_TYPE, f.REGION, f.TENURE_MONTHS, f.MONTHLY_SPEND, f.ORDERS_LAST_30D,
        f.DAYS_SINCE_LAST_ORDER, f.SUPPORT_TICKETS_90D, f.AVG_ORDER_VALUE, f.DISCOUNT_RATE
    ):output_feature_1::FLOAT AS CHURN_PROBABILITY,
    l.CHURNED::FLOAT AS ACTUAL_CHURNED,
    f.PLAN_TYPE, f.REGION, f.TENURE_MONTHS, f.MONTHLY_SPEND, f.ORDERS_LAST_30D,
    f.DAYS_SINCE_LAST_ORDER, f.SUPPORT_TICKETS_90D, f.AVG_ORDER_VALUE, f.DISCOUNT_RATE
FROM {FV_TABLE} f
JOIN {FQ}.CUSTOMER_CHURN l
  ON  l.CUSTOMER_ID       = f.CUSTOMER_ID
  AND l.OBSERVATION_DATE  = f.OBSERVATION_DATE
"""

refresh_sql = f"ALTER DYNAMIC TABLE {FV_TABLE} REFRESH"

with DAG(
    "CHURN_PIPELINE",
    schedule=timedelta(days=1),
    warehouse=WAREHOUSE,
    stage_location=None,
) as dag:
    task_refresh = DAGTask("REFRESH_FEATURES", definition=refresh_sql, warehouse=WAREHOUSE)
    task_score   = DAGTask("SCORE_CUSTOMERS",  definition=score_sql,   warehouse=WAREHOUSE)
    task_monitor = DAGTask("REFRESH_MONITOR_SOURCE", definition=monitor_sql, warehouse=WAREHOUSE)

    task_refresh >> task_score >> task_monitor

dag_op = DAGOperation(root.databases[DB].schemas[SCHEMA])
dag_op.deploy(dag, mode="orreplace")

print(f"Deployed {FQ}.CHURN_PIPELINE (daily, SUSPENDED)")
session.sql(f"SHOW TASKS IN SCHEMA {FQ}").select('"name"', '"state"', '"predecessors"').show()

In [ ]:
# --- Run it once now, without waiting for the schedule ------------------------------
DAG_NAME = "CHURN_PIPELINE"

# The root task may be running from a prior deploy -- suspend it first so we can
# alter child tasks, then resume the whole graph.
session.sql(f"ALTER TASK {FQ}.{DAG_NAME} SUSPEND").collect()

for t in ["REFRESH_FEATURES", "SCORE_CUSTOMERS", "REFRESH_MONITOR_SOURCE"]:
    session.sql(f"ALTER TASK {FQ}.{DAG_NAME}${t} RESUME").collect()
session.sql(f"ALTER TASK {FQ}.{DAG_NAME} RESUME").collect()

session.sql(f"EXECUTE TASK {FQ}.{DAG_NAME}").collect()
print("Triggered CHURN_PIPELINE -- runs asynchronously.\n")

print(session.sql(f"""
    SELECT NAME, STATE, SCHEDULED_TIME, COMPLETED_TIME, ERROR_MESSAGE
    FROM TABLE({DB}.INFORMATION_SCHEMA.TASK_HISTORY(
        DATABASE_NAME => '{DB}',
        SCHEMA_NAME => '{SCHEMA}',
        RESULT_LIMIT => 20
    ))
    ORDER BY SCHEDULED_TIME DESC
    LIMIT 20
""").to_pandas().to_string(index=False))

# Suspend again so the demo does not run nightly. Root first.
#session.sql(f"ALTER TASK {FQ}.{DAG_NAME} SUSPEND").collect()
#for t in ["REFRESH_FEATURES", "SCORE_CUSTOMERS", "REFRESH_MONITOR_SOURCE"]:
#    session.sql(f"ALTER TASK {FQ}.{DAG_NAME}${t} SUSPEND").collect()
#print("\nTasks suspended.")

---
## 10b. Optional: Gated Retraining

Section 10 automated *scoring*, so this section automates *retraining*, which we separated out since this process will have write access to the model serving production.

This is particularly important because sometimes we do get model drift, right? And sometimes model drift doesn't always happen on our schedule- it can interrupt our sleep, vacations, or even fail silently so you only know once you're back in the office! 

This process, thus, enables us to retrain models as new data comes in, test the new model, compare the new model to the old one, and (if it's better) promote the new model to deployment. 

So the pattern is **train, then earn promotion**:

```
REFRESH_FEATURES      ->  refreshes the feature view so training reads current data
RETRAIN_CANDIDATE     ->  trains a new version. Does NOT promote it.
EVALUATE_AND_PROMOTE  ->  compares candidate vs current default on a holdout.
                          Moves the default pointer ONLY if it wins by >= 0.01 AUC.
                          Otherwise the incumbent stays and the rejection is logged.
                          On promotion, also recreates the Model Monitor for the new version.
SCORE_CUSTOMERS       ->  scores with whatever the default now is.
REFRESH_MONITOR_SOURCE
```

Three things make this work:

- **Registry versions are cheap.** A candidate can exist without serving anything.
- **The `default` alias is the switch.** Promotion moves a pointer, it doesn't overwrite a model, and rollback is available if needed.
- **A margin, not just "greater than".** Requiring `>= 0.01` improvement stops the pointer flapping between models that are statistically indistinguishable.

A control table `CHURN_RETRAIN_LOG` records every candidate and whether it was promoted or rejected, which is the audit trail you will want the first time someone asks why the model changed.

Training runs as a **Python stored procedure** on the warehouse, so this still needs no compute
pool. The procedures use `session.get_current_database()` and `session.get_current_schema()` at
runtime rather than hardcoded names, so they work in any account regardless of what you named
your database or schema in §1.

**One honest limitation:** the evaluation holdout here is the same recent 20% used earlier. In a
real system the holdout should be data newer than *both* models' training windows, otherwise the
incumbent is being judged on data it has already seen and the comparison flatters the candidate.

**Model Monitor and promotion:** `CREATE MODEL MONITOR` requires an explicit version name — it
does not support `VERSION DEFAULT`. So when `EVALUATE_AND_PROMOTE` promotes a new version, it
drops and recreates the monitor pointing to that version. This is expected behaviour, not a bug;
accumulated historical metrics are lost on recreation, which is a trade-off worth noting.

In [ ]:
# --- Control table: the audit trail for every retrain attempt -----------------------
session.sql(f"""
CREATE TABLE IF NOT EXISTS {FQ}.CHURN_RETRAIN_LOG (
    CANDIDATE_VERSION  STRING,
    CREATED_AT         TIMESTAMP_NTZ,
    STATUS             STRING,        -- PENDING | PROMOTED | REJECTED
    CANDIDATE_AUC      FLOAT,
    INCUMBENT_VERSION  STRING,
    INCUMBENT_AUC      FLOAT,
    NOTE               STRING
)
""").collect()

# --- RETRAIN_CANDIDATE: train a new version, do NOT promote it. -------------------
# Note that we do use a fixed seed here again so that this demo's results are reproducible!!
retrain_proc = f"""
CREATE OR REPLACE PROCEDURE {FQ}.RETRAIN_CANDIDATE()
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python','snowflake-ml-python','scikit-learn','pandas','numpy')
HANDLER = 'main'
AS
$$
from datetime import datetime

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from snowflake.ml.model import task as ml_task
from snowflake.ml.registry import Registry

CATEGORICAL = ["PLAN_TYPE", "REGION"]
NUMERIC = ["TENURE_MONTHS", "MONTHLY_SPEND", "ORDERS_LAST_30D", "DAYS_SINCE_LAST_ORDER",
           "SUPPORT_TICKETS_90D", "AVG_ORDER_VALUE", "DISCOUNT_RATE"]
FEATURES = CATEGORICAL + NUMERIC
TARGET = "CHURNED"
MODEL_NAME = "CHURN_CLASSIFIER"


def main(session):
    db     = session.get_current_database().strip('"')
    schema = session.get_current_schema().strip('"')
    fq     = f"{{db}}.{{schema}}"

    df = session.table(f"{{fq}}.CUSTOMER_CHURN").to_pandas()
    df["OBSERVATION_DATE"] = pd.to_datetime(df["OBSERVATION_DATE"])

    # Chronological split -- same discipline as the interactive training step.
    cutoff = df["OBSERVATION_DATE"].quantile(0.80)
    train  = df[df["OBSERVATION_DATE"] <= cutoff]
    X, y   = train[FEATURES], train[TARGET]

    pipe = Pipeline([
        ("pre", ColumnTransformer([
            ("num", Pipeline([("impute", SimpleImputer(strategy="median"))]), NUMERIC),
            ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
        ])),
        # No random_state: each retrain should be an independent model.
        ("model", HistGradientBoostingClassifier(max_iter=200, learning_rate=0.08)),
    ])
    pipe.fit(X, y)

    version = "V_" + datetime.utcnow().strftime("%Y%m%d_%H%M%S")
    reg = Registry(session=session, database_name=db, schema_name=schema)

    # Pass a Snowpark DataFrame (not pandas) as sample_input_data so Snowflake can
    # record the upstream lineage edge: candidate version → CUSTOMER_CHURN.
    # Inside a stored procedure you can't reference notebook variables; build a fresh
    # Snowpark DF from session.table() instead.
    sample_sf = session.table(f"{{fq}}.CUSTOMER_CHURN").select(FEATURES).limit(100)

    reg.log_model(
        model=pipe,
        model_name=MODEL_NAME,
        version_name=version,
        sample_input_data=sample_sf,
        task=ml_task.Task.TABULAR_BINARY_CLASSIFICATION,
        comment="automated retrain candidate -- awaiting gate evaluation",
    )
    # NOTE: no model.default = version here. That happens only in EVALUATE_AND_PROMOTE.

    session.sql(
        f"INSERT INTO {{fq}}.CHURN_RETRAIN_LOG "
        f"(CANDIDATE_VERSION, CREATED_AT, STATUS) "
        f"SELECT '{{version}}', CURRENT_TIMESTAMP(), 'PENDING'"
    ).collect()

    return f"Trained candidate {{version}} on {{len(X)}} rows (awaiting evaluation)"
$$
"""
session.sql(retrain_proc).collect()
print("Created procedure RETRAIN_CANDIDATE")

In [ ]:
# --- EVALUATE_AND_PROMOTE: gate that controls which model version serves production. ---
# Same notes as RETRAIN_CANDIDATE: literal string body, session methods for schema.
evaluate_proc = f"""
CREATE OR REPLACE PROCEDURE {FQ}.EVALUATE_AND_PROMOTE()
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python','snowflake-ml-python','scikit-learn','pandas','numpy')
HANDLER = 'main'
AS
$$
import pandas as pd
from sklearn.metrics import roc_auc_score

from snowflake.ml.registry import Registry

MODEL_NAME   = "CHURN_CLASSIFIER"
MONITOR_NAME = "CHURN_MONITOR"
TARGET       = "CHURNED"
WAREHOUSE    = "ML_DEMO_WH"   # used only if recreating the monitor

# Require a real improvement, not noise. Without a margin the pointer flaps between
# models that are statistically indistinguishable.
MIN_IMPROVEMENT = 0.01


def _auc(mv, holdout_sdf):
    # Score and read the label from THE SAME DATAFRAME.
    # Pulling y_true via a separate query would rely on two queries returning rows in
    # the same order, which Snowflake does not guarantee -- a silent misalignment that
    # would corrupt the metric. Keep them together.
    scored = mv.run(holdout_sdf, function_name="predict_proba").to_pandas()
    return float(roc_auc_score(scored[TARGET], scored["output_feature_1"]))


def main(session):
    db     = session.get_current_database().strip('"')
    schema = session.get_current_schema().strip('"')
    fq     = f"{{db}}.{{schema}}"

    pending = session.sql(
        f"SELECT CANDIDATE_VERSION FROM {{fq}}.CHURN_RETRAIN_LOG "
        f"WHERE STATUS = 'PENDING' ORDER BY CREATED_AT DESC LIMIT 1"
    ).collect()
    if not pending:
        return "No pending candidate -- nothing to evaluate."
    candidate = pending[0][0]

    reg   = Registry(session=session, database_name=db, schema_name=schema)
    model = reg.get_model(MODEL_NAME)
    incumbent = model.default.version_name

    if incumbent == candidate:
        return f"Candidate {{candidate}} is already the default -- nothing to do."

    # Holdout: the most recent 20% of observations.
    # OBSERVATION_DATE is stored as NUMBER(38,0) epoch nanoseconds by write_pandas;
    # cast with scale=9 before comparing to a date string to avoid a type mismatch.
    df = session.table(f"{{fq}}.CUSTOMER_CHURN").to_pandas()
    df["OBSERVATION_DATE"] = pd.to_datetime(df["OBSERVATION_DATE"])
    cutoff = df["OBSERVATION_DATE"].quantile(0.80).strftime("%Y-%m-%d")

    holdout_sdf = session.table(f"{{fq}}.CUSTOMER_CHURN").filter(
        f"TO_DATE(TO_TIMESTAMP_NTZ(OBSERVATION_DATE, 9)) > '{{cutoff}}'"
    )

    cand_auc = _auc(model.version(candidate), holdout_sdf)
    inc_auc  = _auc(model.version(incumbent), holdout_sdf)
    delta    = cand_auc - inc_auc

    promote = delta >= MIN_IMPROVEMENT
    if promote:
        model.default = candidate   # moving a pointer -- rollback is moving it back
        status, note = "PROMOTED", f"beat incumbent by {{delta:+.4f}}"

        # Model Monitor is coupled to a specific version (CREATE MODEL MONITOR requires
        # VERSION = <explicit>; DEFAULT is not supported). Recreate it for the new default
        # so monitoring continues to track what is actually serving.
        session.sql(f"DROP MODEL MONITOR IF EXISTS {{fq}}.{{MONITOR_NAME}}").collect()
        session.sql(f'''
            CREATE MODEL MONITOR {{fq}}.{{MONITOR_NAME}}
            WITH
                MODEL    = {{fq}}.{{MODEL_NAME}}
                VERSION  = {{candidate}}
                FUNCTION = PREDICT_PROBA
                SOURCE   = {{fq}}.CHURN_MONITOR_SOURCE
                BASELINE = {{fq}}.CHURN_MONITOR_BASELINE
                ID_COLUMNS               = ('CUSTOMER_ID')
                TIMESTAMP_COLUMN         = 'PREDICTION_TIMESTAMP'
                PREDICTION_SCORE_COLUMNS = ('CHURN_PROBABILITY')
                ACTUAL_CLASS_COLUMNS     = ('ACTUAL_CHURNED')
                WAREHOUSE          = {{WAREHOUSE}}
                REFRESH_INTERVAL   = '1 hour'
                AGGREGATION_WINDOW = '7 days'
            ''').collect()
    else:
        status, note = "REJECTED", f"delta {{delta:+.4f}} below margin {{MIN_IMPROVEMENT}}"

    session.sql(
        f"UPDATE {{fq}}.CHURN_RETRAIN_LOG SET "
        f"STATUS = '{{status}}', CANDIDATE_AUC = {{cand_auc}}, "
        f"INCUMBENT_VERSION = '{{incumbent}}', INCUMBENT_AUC = {{inc_auc}}, "
        f"NOTE = '{{note}}' "
        f"WHERE CANDIDATE_VERSION = '{{candidate}}'"
    ).collect()

    return (f"{{status}}: candidate {{candidate}} AUC {{cand_auc:.4f}} vs "
            f"incumbent {{incumbent}} AUC {{inc_auc:.4f}} ({{note}})")
$$
"""
session.sql(evaluate_proc).collect()
print("Created procedure EVALUATE_AND_PROMOTE")

In [ ]:
# --- Try the gate by hand before trusting it to a schedule --------------------------
# Expect REJECTED. The candidate is trained on the same data with the same settings as V1,
# so it should land within noise of the incumbent and fail to clear the 0.05 margin.
# A gate that rejects an identical model is a gate that works.

print(session.call(f"{FQ}.RETRAIN_CANDIDATE"))
print(session.call(f"{FQ}.EVALUATE_AND_PROMOTE"))

print("\nAudit trail:")
session.sql(f"""
    SELECT CANDIDATE_VERSION, STATUS, CANDIDATE_AUC,
           INCUMBENT_VERSION, INCUMBENT_AUC, NOTE, CREATED_AT
    FROM {FQ}.CHURN_RETRAIN_LOG
    ORDER BY CREATED_AT DESC
""").show()

print("Model versions now in the registry (note which one is default):")
reg.get_model(MODEL_NAME).show_versions()

In [ ]:
# --- Redeploy the DAG with retraining in front of scoring ---------------------------
# Weekly, not daily: retraining is computationally intensive and models rarely decay that fast without real-time inference.
# Scoring stays effectively daily in a real setup- you would normally split these into
# two graphs on different schedules, but it's kept as one here so the ordering is visible.

with DAG(
    "CHURN_PIPELINE",
    schedule=timedelta(days=7),
    warehouse=WAREHOUSE,
    stage_location=None,
) as dag_v2:
    t_refresh = DAGTask("REFRESH_FEATURES", definition=refresh_sql, warehouse=WAREHOUSE)
    t_retrain = DAGTask("RETRAIN_CANDIDATE",
                        definition=f"CALL {FQ}.RETRAIN_CANDIDATE()", warehouse=WAREHOUSE)
    t_gate    = DAGTask("EVALUATE_AND_PROMOTE",
                        definition=f"CALL {FQ}.EVALUATE_AND_PROMOTE()", warehouse=WAREHOUSE)
    t_score   = DAGTask("SCORE_CUSTOMERS", definition=score_sql, warehouse=WAREHOUSE)
    t_monitor = DAGTask("REFRESH_MONITOR_SOURCE", definition=monitor_sql, warehouse=WAREHOUSE)

    t_refresh >> t_retrain >> t_gate >> t_score >> t_monitor

dag_op.deploy(dag_v2, mode="orreplace")
print(f"Redeployed {FQ}.CHURN_PIPELINE with gated retraining (weekly, SUSPENDED)")
session.sql(f"SHOW TASKS IN SCHEMA {FQ}").select('"name"', '"state"', '"predecessors"').show()

In [ ]:
# --- Run the V2 pipeline (with retraining) once to test it --------------------------
# Same pattern as the §10 cell, extended to cover all 5 child tasks.
#
# Also refreshes the feature view first: RETRAIN_CANDIDATE trains on it, and
# SCHEDULER = DISABLE means it only refreshes when we ask.

session.sql(f"ALTER DYNAMIC TABLE {FV_TABLE} REFRESH").collect()
print("Feature view refreshed.")

session.sql(f"ALTER TASK {FQ}.{DAG_NAME} SUSPEND").collect()

for t in ["REFRESH_FEATURES", "RETRAIN_CANDIDATE", "EVALUATE_AND_PROMOTE",
          "SCORE_CUSTOMERS", "REFRESH_MONITOR_SOURCE"]:
    session.sql(f"ALTER TASK {FQ}.{DAG_NAME}${t} RESUME").collect()
session.sql(f"ALTER TASK {FQ}.{DAG_NAME} RESUME").collect()

session.sql(f"EXECUTE TASK {FQ}.{DAG_NAME}").collect()
print("Triggered V2 CHURN_PIPELINE (retrain + gate + score) -- runs asynchronously.\n")

print(session.sql(f"""
    SELECT NAME, STATE, SCHEDULED_TIME, COMPLETED_TIME, ERROR_MESSAGE
    FROM TABLE({DB}.INFORMATION_SCHEMA.TASK_HISTORY(
        DATABASE_NAME => '{DB}',
        SCHEMA_NAME => '{SCHEMA}',
        RESULT_LIMIT => 20
    ))
    ORDER BY SCHEDULED_TIME DESC
    LIMIT 20
""").to_pandas().to_string(index=False))

# Suspend again -- root first.
session.sql(f"ALTER TASK {FQ}.{DAG_NAME} SUSPEND").collect()
for t in ["REFRESH_FEATURES", "RETRAIN_CANDIDATE", "EVALUATE_AND_PROMOTE",
          "SCORE_CUSTOMERS", "REFRESH_MONITOR_SOURCE"]:
    session.sql(f"ALTER TASK {FQ}.{DAG_NAME}${t} SUSPEND").collect()
print("\nTasks suspended.")

---
## 11. ML Lineage

Lineage is the graph of what fed what: source table → feature view → dataset → model.

You do not have to declare any of it. Because we used the Feature Store, Datasets, and the Registry
together, Snowflake recorded the edges as we went. This section just reads them back.

Two questions it answers, both of which are painful without it:

- *"Someone wants to change `CUSTOMER_CHURN`. What breaks?"* Just look downstream!
- *"This model is making strange predictions. What was it trained on?"* Go look upstream!

The second one is also the compliance question, phrased more politely.

In [ ]:
# Lineage can take a few minutes to populate after objects are created.
# Empty results here mean "not indexed yet", not "no lineage".

print("DOWNSTREAM from CUSTOMER_CHURN -- what would break if we changed this table?")
session.sql(f"""
    SELECT SOURCE_OBJECT_NAME, SOURCE_OBJECT_DOMAIN,
           TARGET_OBJECT_NAME, TARGET_OBJECT_DOMAIN, DISTANCE
    FROM TABLE(SNOWFLAKE.CORE.GET_LINEAGE(
        '{FQ}.CUSTOMER_CHURN', 'TABLE', 'DOWNSTREAM', 4
    ))
    ORDER BY DISTANCE
""").show(20)

print(f"\nUPSTREAM from {MODEL_NAME} -- what was this model actually trained on?")
session.sql(f"""
    SELECT SOURCE_OBJECT_NAME, SOURCE_OBJECT_DOMAIN,
           TARGET_OBJECT_NAME, TARGET_OBJECT_DOMAIN, DISTANCE
    FROM TABLE(SNOWFLAKE.CORE.GET_LINEAGE(
        object_name => '{FQ}.{MODEL_NAME}',
        object_domain => 'MODULE',
        direction => 'UPSTREAM',
        max_distance => 4,
        object_version => '{MODEL_VERSION}'
    ))
    ORDER BY DISTANCE
""").show(20)

---
## 12. Unified Observability

Now, we've made quite a bit of progress today! Before we wrap up, this cell brings three separate history sources into one view: Task history (from the DAG), Feature View refresh history (from the dynamic table), and Model Monitor status. It isn't a true "single pane" because the three components run on independent schedulers and Snowflake doesn't expose a single unified view — but this query gives you the next-best thing.

After this we have one more optional cell to set up alerting, then a summary and cleanup.

In [ ]:
import pandas as pd

# Build the feature view's physical name from the Python object rather than hardcoding it,
# so this stays correct if you register a V2 later.
FV_NAME = customer_fv.fully_qualified_name().split(".")[-1]   # e.g. CUSTOMER_FEATURES$V1

# Tasks
tasks = session.sql(f"""
    SELECT 'TASK' AS COMPONENT, NAME, STATE, COMPLETED_TIME AS LAST_RUN, ERROR_MESSAGE
    FROM TABLE({DB}.INFORMATION_SCHEMA.TASK_HISTORY(DATABASE_NAME => '{DB}', SCHEMA_NAME => '{SCHEMA}', RESULT_LIMIT => 50))
    ORDER BY SCHEDULED_TIME DESC
    LIMIT 20
""").to_pandas()

# Dynamic table (the feature view)
dt = session.sql(f"""
    SELECT 'DYNAMIC_TABLE' AS COMPONENT, NAME, STATE, DATA_TIMESTAMP AS LAST_RUN, STATE_MESSAGE AS ERROR_MESSAGE
    FROM TABLE(INFORMATION_SCHEMA.DYNAMIC_TABLE_REFRESH_HISTORY())
    WHERE NAME = '{FV_NAME}'
    ORDER BY DATA_TIMESTAMP DESC
    LIMIT 5
""").to_pandas()

# Monitor -- DESCRIBE gives us state and last-updated timestamp
desc = session.sql(f"DESCRIBE MODEL MONITOR {FQ}.CHURN_MONITOR").collect()[0].as_dict()
mon = pd.DataFrame([{
    "COMPONENT": "MODEL_MONITOR",
    "NAME": "CHURN_MONITOR",
    "STATE": desc.get("monitor_state"),
    "LAST_RUN": desc.get("aggregation_last_data_timestamp"),
    "ERROR_MESSAGE": desc.get("aggregation_last_error"),
}])

pipeline = pd.concat([tasks, dt, mon], ignore_index=True)
pipeline["LAST_RUN"] = pd.to_datetime(pipeline["LAST_RUN"], errors="coerce")

# Keep only the most recent event per component + name
pipeline = (pipeline
    .sort_values("LAST_RUN", ascending=False)
    .drop_duplicates(subset=["COMPONENT", "NAME"], keep="first")
    .reset_index(drop=True)
)
print(pipeline.to_string(index=False))

In [ ]:
# Before running this cell:
# 1. Verify your email address in Snowsight: Profile → Email → click the verification link.
# 2. Confirm ACCOUNTADMIN (or a role with CREATE INTEGRATION) can run the first statement.
#
# SYSTEM$SEND_EMAIL can only send to verified email addresses of users in the same account.
# Replace YOUR_VERIFIED_EMAIL with your actual Snowflake-account email, after verifying it.

EMAIL = "your.email@example.com"   # <-- change this before running

session.sql(f"""
    CREATE OR REPLACE NOTIFICATION INTEGRATION ml_demo_email
        TYPE  = EMAIL
        ENABLED = TRUE
""").collect()
print("Created notification integration ml_demo_email")

# Grant SYSADMIN usage so the scheduled alert task can call SYSTEM$SEND_EMAIL.
session.sql("GRANT USAGE ON INTEGRATION ml_demo_email TO ROLE SYSADMIN").collect()

# Alert: fires if CHURN_PREDICTIONS hasn't been refreshed in the last 2 days.
# OBSERVATION_DATE is stored as NUMBER(38,0) epoch nanoseconds, so we cast before comparing.
session.sql(f"""
    CREATE OR REPLACE ALERT {FQ}.PIPELINE_STALE
        WAREHOUSE = {WAREHOUSE}
        SCHEDULE  = '60 MINUTE'
        IF (EXISTS (
            SELECT 1 FROM {FQ}.CHURN_PREDICTIONS
            HAVING MAX(TO_DATE(TO_TIMESTAMP_NTZ(OBSERVATION_DATE, 9)))
                   < DATEADD('day', -2, CURRENT_DATE())
        ))
        THEN CALL SYSTEM$SEND_EMAIL(
            'ml_demo_email',
            '{EMAIL}',
            'Churn pipeline stale',
            'CHURN_PREDICTIONS has not updated in over 2 days. Check TASK_HISTORY.'
        )
""").collect()
print(f"Created alert {FQ}.PIPELINE_STALE")

# Alerts are created suspended. Resume when you are ready for live monitoring.
session.sql(f"ALTER ALERT {FQ}.PIPELINE_STALE RESUME").collect()

---
## Summary

### Objects created

| Object | Type | Purpose |
|--------|------|---------|
| `ML_DEMO` / `ML_DEMO.CHURN` | Database / Schema | Container for everything |
| `ML_DEMO_WH` | Warehouse | XSMALL compute |
| `CUSTOMER_CHURN` | Table | 50K labelled rows |
| `CUSTOMER_FEATURES/V1` | **Feature View** | Feature definitions (dynamic table) |
| `CHURN_TRAINING_DATA/V1` | **Dataset** | Immutable training snapshot |
| `CHURN_EXPERIMENT_V1` | **Experiment** | Training runs logged here |
| `CHURN_CLASSIFIER/V1` | **Model** | Versioned model, set as default |
| `CHURN_PREDICTIONS` | Table | Batch scoring output |
| `CHURN_MONITOR_SOURCE` / `_BASELINE` | Tables | Monitoring inputs |
| `CHURN_MONITOR` | **Model Monitor** | Drift + performance |
| `RETRAIN_CANDIDATE` / `EVALUATE_AND_PROMOTE` | **Procedures** | Gated retraining (§10b) |
| `CHURN_RETRAIN_LOG` | Table | Audit trail: every candidate, promoted or rejected |
| `CHURN_PIPELINE` + 5 children | **Tasks** | Automation (suspended) |
| `PIPELINE_STALE` | **Alert** | Stale-data notification (suspended) |
| `ml_demo_email` | **Notification Integration** | Email delivery for the alert |

### The habits worth keeping

1. **Write the prediction sentence first.** It defines your label and your feature cutoff, which is what keeps leakage out.
2. **Always build a baseline.** A score with nothing to compare it against is not information.
3. **Split before you preprocess, and split by time.** Both mistakes inflate your score and hide real decay.
4. **Put preprocessing inside the pipeline you register.** Train/serve skew stops being possible rather than being something you remember to avoid.
5. **Register early.** Versioning, schema, metrics, and SQL access all arrive free, and none of it can be retrofitted onto a pickle file on your laptop.
6. **Never let retraining promote itself.** Train into a new version, make it earn the `default` pointer against the incumbent, and log the verdict either way.

### What was deliberately left out

- **HPO / hyperparameter tuning**: tuning a pipeline you have not yet validated is wasted compute. Add it once the pipeline is boring.
- **Real-time serving (SPCS)**: only needed when a user waits on a single prediction. Requires a compute pool.
- **ML Jobs (`@remote`)**: for when training outgrows a warehouse or needs a GPU. Not a beginner concern.

None of the three is required for an automated pipeline. Everything here, including retraining, runs on a warehouse via the Task Graph.

### Natural next steps

- Add `mv.explain()` for SHAP values, which provides per-prediction feature attributions
- Split the DAG in two: score daily, retrain weekly, on separate schedules
- Wire a **notification integration** to the `REJECTED` path, so a failing gate tells someone
- Give the gate a holdout newer than both models' training windows (see the caveat in §10b)

In [ ]:
# --- CLEANUP ------------------------------------------------------------------------
# Uncomment and run to remove everything this notebook created.
# Suspend the root task first -- a running task blocks its own DROP.

cleanup = [
     f"ALTER TASK IF EXISTS {FQ}.CHURN_PIPELINE SUSPEND",          # root first
     f"ALTER TASK IF EXISTS {FQ}.EVALUATE_AND_PROMOTE SUSPEND",
     f"ALTER TASK IF EXISTS {FQ}.RETRAIN_CANDIDATE SUSPEND",
     f"ALTER TASK IF EXISTS {FQ}.SCORE_CUSTOMERS SUSPEND",
     f"ALTER TASK IF EXISTS {FQ}.REFRESH_FEATURES SUSPEND",
     f"ALTER TASK IF EXISTS {FQ}.REFRESH_MONITOR_SOURCE SUSPEND",
     f"ALTER ALERT IF EXISTS {FQ}.PIPELINE_STALE SUSPEND",
     f"DROP MODEL MONITOR IF EXISTS {FQ}.CHURN_MONITOR",
     f"DROP DATABASE IF EXISTS {DB}",      # cascades: tables, model, monitor, tasks, procedures
     f"DROP WAREHOUSE IF EXISTS {WAREHOUSE}",
     "DROP NOTIFICATION INTEGRATION IF EXISTS ml_demo_email",  # account-level, not in DB
]

for stmt in cleanup:
    session.sql(stmt).collect()
    print(f"OK: {stmt}")

print("Nothing dropped." if not cleanup else "Cleanup complete.")